# Архівна чернетка

Цей ноутбук збережено лише для історії. Його замінюють етапи 09–13; не використовуйте його як джерело фінальних метрик.


In [ ]:
import os
import time
from pathlib import Path

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pywt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor

# Завантаження даних та моделей

## Дані

In [ ]:
train = pd.read_csv("/content/drive/MyDrive/predictive_maintenance/train_main_26.csv")
test = pd.read_csv("/content/drive/MyDrive/predictive_maintenance/test_main_26.csv")
rul = pd.read_csv('/content/drive/MyDrive/predictive_maintenance/RUL_FD001.txt', sep=r"\s+", header=None)

In [ ]:
print("train:")
print(train.columns.tolist())

print("\ntest:")
print(test.columns.tolist())

print("\nПерші рядки train:")
display(train.head())

## Моделі

### XGB

In [ ]:
from xgboost import XGBRegressor

XGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.012074507989018332,
    "max_depth": 4,
    "min_child_weight": 3.2298041995296414,
    "subsample": 0.7900204897773602,
    "colsample_bytree": 0.9223090698240903,
    "reg_alpha": 0.7459635164536705,
    "reg_lambda": 4.416722489003329,
    "gamma": 0.2695542352154229,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

xgb_model = XGBRegressor(**XGB_PARAMS)

print("XGBoost-модель створена")
print("Параметри:", xgb_model.get_params())

### Обробка даних для CCN та LSTM

In [ ]:
class RULDataPreprocessor:
    def __init__(self, base_features):
        self.base_features = list(base_features)
        self.missing_features = []
        self.indicator_features = []
        self.feature_columns = None
        self.scaler = StandardScaler()
        self.is_fitted = False

    def fit_transform(self, train_df):
        train = train_df.copy()
        self.missing_features = [c for c in self.base_features if train[c].isna().any()]
        self.indicator_features = [f"{c}_was_missing" for c in self.missing_features]

        for c, ind in zip(self.missing_features, self.indicator_features):
            train[ind] = train[c].isna().astype("int8")

        train[self.base_features] = train[self.base_features].fillna(0)
        self.feature_columns = self.base_features + self.indicator_features
        self._validate(train, "train")

        train[self.feature_columns] = self.scaler.fit_transform(train[self.feature_columns])
        self.is_fitted = True
        return train

    def transform(self, df):
        if not self.is_fitted:
            raise RuntimeError("Спочатку fit_transform() на train.")
        result = df.copy()

        # індикатори — тільки для колонок, у яких були пропуски на train
        for c, ind in zip(self.missing_features, self.indicator_features):
            result[ind] = result[c].isna().astype("int8")

        result[self.base_features] = result[self.base_features].fillna(0)
        self._validate(result, "transform")
        result[self.feature_columns] = self.scaler.transform(result[self.feature_columns])
        return result

    @staticmethod
    def _validate(df, name):
        values = df.select_dtypes(include=[np.number]).to_numpy()
        if not np.isfinite(values).all():
            raise ValueError(f"{name}: знайдено NaN або нескінченні значення.")

In [ ]:
def create_windows(df, feature_cols, window_size):
    X, y = [], []

    for _, engine in df.groupby("unit_id", sort=False):
        engine = engine.sort_values("cycle")

        features = engine[feature_cols].to_numpy(
            dtype=np.float32
        )
        rul = engine["RUL"].to_numpy(dtype=np.float32)

        for end in range(window_size, len(engine) + 1):
            start = end - window_size

            X.append(features[start:end])
            y.append(rul[end - 1])

    if not X:
        raise ValueError(
            f"Не вдалося створити вікна довжиною {window_size}."
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
    )

In [ ]:
def create_rul_dataloader(
    df,
    feature_cols,
    window_size,
    batch_size=64,
    shuffle=False,
):
    X, y = create_windows(
        df,
        feature_cols,
        window_size=window_size,
    )

    X_tensor = torch.from_numpy(X)
    y_tensor = torch.from_numpy(y).unsqueeze(1)

    dataset = TensorDataset(X_tensor, y_tensor)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )

    return loader, X.shape

In [ ]:
def create_official_test_windows(df, feature_cols, window_size=80):
    X = []

    for _, engine in df.groupby("unit_id", sort=True):
        engine = engine.sort_values("cycle")

        features = engine[feature_cols].to_numpy(dtype=np.float32)

        # Беремо останні 80 циклів кожного двигуна.
        # Якщо циклів менше — доповнюємо початок першим спостереженням.
        if len(features) < window_size:
            padding = np.repeat(
                features[[0]],
                window_size - len(features),
                axis=0,
            )
            features = np.concatenate([padding, features], axis=0)

        X.append(features[-window_size:])

    return np.asarray(X, dtype=np.float32)

# Cloud

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

### CNN

In [ ]:
class RULCNNNoPooling(nn.Module):
    def __init__(self, in_channels, dropout_p=0.3):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=32,
                kernel_size=3
            ),
            nn.ReLU(),

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=3
            ),
            nn.ReLU()
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # Вхід: [batch, window, features]
        # Conv1d: [batch, channels, sequence_length]
        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x.squeeze(-1)

In [ ]:
CNN_CHECKPOINT_PATH = ("/content/drive/MyDrive/predictive_maintenance/cnn/cnn_fd001_best.pt")

checkpoint = torch.load(
    CNN_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)

config = checkpoint["config"]

cnn_model = RULCNNNoPooling(
    in_channels=config["in_channels"],
    dropout_p=config["dropout_p"],
)

cnn_model.load_state_dict(checkpoint["model_state_dict"])

cnn_model = cnn_model.to(device)
cnn_model.eval()

print("CNN завантажена")
print("Best epoch:", checkpoint["best_epoch"])
print("Best validation MAE:", checkpoint["best_val_mae"])

### LSTM

In [ ]:
class LSTMRegressor(nn.Module):
  def __init__(self, input_size= 25, hidden_size= 64):
    super().__init__()

    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=1,
        batch_first=True
    )

    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):

    output, (h_n, c_n) = self.lstm(x)
    last_hidden = h_n[-1]
    prediction = self.fc(last_hidden)

    return prediction

In [ ]:
MODEL_PATH_LSTM = Path(
    "/content/drive/MyDrive/predictive_maintenance/lstm/lstm_fd001_best.pt"
)
checkpoint = torch.load(
    MODEL_PATH_LSTM,
    map_location="cpu",
    weights_only=True,
)

lstm_model = LSTMRegressor(
    input_size=checkpoint["input_size"],
    hidden_size=checkpoint["hidden_size"],
)

# Завантажуємо навчені ваги
lstm_model.load_state_dict(checkpoint["model_state_dict"])

lstm_model = lstm_model.to(device)
lstm_model.eval()

print("Модель завантажена")
print("Window size:", checkpoint["window_size"])
print("Input size:", checkpoint["input_size"])
print("Best epoch:", checkpoint["best_epoch"])
print("Validation MAE:", checkpoint["val_mae"])
print("Validation RMSE:", checkpoint["val_rmse"])

In [ ]:
len(train.columns.tolist())

# Тест

In [ ]:
print("CNN очікує ознак:", config["in_channels"])
print("LSTM очікує ознак:", checkpoint["input_size"])
print("Підготовлено ознак:", X_val_tensor.shape[-1])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_rul_model(model, X_test, y_test, model_name, device):
    model.eval()

    X = X_test.to(device)

    # CNN: [batch, window, features] -> [batch, features, window]
    if isinstance(model, RULCNNNoPooling):
        X = X.transpose(1, 2)

    with torch.inference_mode():
        predictions = model(X).squeeze(-1).cpu().numpy()

    y_true = np.asarray(y_test, dtype=np.float32)

    results = {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "R2": r2_score(y_true, predictions),
        "Predictions": predictions
    }

    return results

In [ ]:
cnn_results = evaluate_rul_model(
    cnn_model, X_test_tensor, y_test, "CNN", device
)

lstm_results = evaluate_rul_model(
    lstm_model, X_test_tensor, y_test, "LSTM", device
)

results_df = pd.DataFrame([
    {k: v for k, v in cnn_results.items() if k != "Predictions"},
    {k: v for k, v in lstm_results.items() if k != "Predictions"}
])

print(results_df.to_string(index=False))

In [ ]:
comparison_df = pd.DataFrame({
    "Engine": np.arange(1, 101),
    "Actual_RUL": y_test,
    "CNN_Predicted_RUL": cnn_results["Predictions"],
    "LSTM_Predicted_RUL": lstm_results["Predictions"]
})

display(comparison_df)